# Tsinghua University AI Research Analysis

Exploratory Data Analysis of AI publications from Tsinghua University using OpenAlex API.

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Environment ready")

In [ ]:
# Load processed data
df = pd.read_parquet("../data/processed/openalex_clean.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Missing values
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Publications per year
yearly = df.groupby('publication_year').size().reset_index(name='count')
fig = px.line(yearly, x='publication_year', y='count', markers=True,
              title='Publications per Year')
fig.show()

In [ ]:
# Citations distribution
fig = px.histogram(df[df['cited_by_count'] <= 200], x='cited_by_count', nbins=50,
                   title='Citation Distribution (capped at 200)')
fig.show()

In [ ]:
# Top cited papers
top_papers = df.nlargest(10, 'cited_by_count')[['title', 'publication_year', 'cited_by_count', 'venue_name', 'concept_group']]
top_papers

In [ ]:
# Papers by concept group
area_counts = df['concept_group'].value_counts().reset_index()
area_counts.columns = ['concept_group', 'count']
fig = px.bar(area_counts, x='concept_group', y='count',
             title='Papers by Research Area')
fig.show()

In [ ]:
# International collaboration rate
intl_rate = df['is_international'].mean()
print(f"International collaboration rate: {intl_rate:.1%}")

intl_by_year = df.groupby('publication_year')['is_international'].mean().reset_index()
fig = px.line(intl_by_year, x='publication_year', y='is_international',
              title='International Collaboration Rate Over Time',
              markers=True)
fig.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['cited_by_count', 'authors_count', 'paper_age',
                'citations_per_year', 'countries_count', 'abstract_length']
corr = df[numeric_cols].corr()
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                title='Correlation Matrix',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.show()

In [ ]:
# Interactive scatter: Year vs Citations
df_scatter = df[df['cited_by_count'] <= 500].copy()
fig = px.scatter(df_scatter, x='publication_year', y='cited_by_count',
                 color='concept_group', size='authors_count',
                 hover_data=['title', 'venue_name', 'countries'],
                 title='Year vs Citations by Research Area',
                 trendline='ols')
fig.show()

In [ ]:
# Treemap: Concept hierarchy
treemap_df = df.groupby(['concept_group', 'primary_concept']).size().reset_index(name='count')
fig = px.treemap(treemap_df, path=['concept_group', 'primary_concept'], values='count',
                 title='Research Area Hierarchy')
fig.show()